# Silver Layer Transformation 

In the Silver layer, the Bronze datasets are cleaned and standardized to ensure data consistency and usability for downstream analytics. String fields are trimmed, special characters like | are removed, and placeholder values such as N/A, NA, or empty strings are replaced with NULL. Country names are normalized; for example, UK, U.K., and misspelled variants like United Kindom are standardized to United Kingdom. Monetary fields like unit_price, cost, and sales are converted from strings with currency symbols into numeric values, while a separate currency column stores the currency type. Duplicate records are eliminated using the appropriate primary keys, and an ingestion_timestamp column is added to record when the data was processed in the Silver layer.

# Silver Table (Region)

# SQL check 

In [0]:
%sql
SELECT * FROM workspace.default.bronze_region;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.silver_region;

# Silver Table (Region) with pyspark

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
bronze_region = spark.table("workspace.default.bronze_region")

# Trim all string columns
silver_region = bronze_region
for c, t in silver_region.dtypes:
    if t == "string":
        silver_region = silver_region.withColumn(c, F.trim(F.col(c)))

# Remove bars/pipes and clean " - " separator (safe cleaning)
for c, t in silver_region.dtypes:
    if t == "string":
        silver_region = silver_region.withColumn(c, F.regexp_replace(F.col(c), r"\|", ""))
        silver_region = silver_region.withColumn(c, F.regexp_replace(F.col(c), r"\s-\s", " "))

# Standardize country names USA to United States
if "country" in silver_region.columns:
    silver_region = silver_region.withColumn(
        "country",
        F.when(F.upper(F.col("country")) == "USA", "United States").otherwise(F.col("country"))
    )

# Remove null keys plus duplicates
silver_region = (
    silver_region
    .filter(F.col("salesterritorykey").isNotNull())
    .dropDuplicates(["salesterritorykey"])
)

# Save as Silver Delta table 
silver_region.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("workspace.default.silver_region")

# Preview
display(silver_region)

# Silver Table (Salespersonregion)

In [0]:
%sql SELECT * FROM workspace.default.bronze_salesperson_region;

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
bronze_salesperson_region = spark.table("workspace.default.bronze_salesperson_region")

# Trim all string columns
silver_salesperson_region = bronze_salesperson_region
for c, t in silver_salesperson_region.dtypes:
    if t == "string":
        silver_salesperson_region = silver_salesperson_region.withColumn(c, F.trim(F.col(c)))

# Remove bars/pipes and clean " - " separator (safe cleaning)
for c, t in silver_salesperson_region.dtypes:
    if t == "string":
        silver_salesperson_region = silver_salesperson_region.withColumn(c, F.regexp_replace(F.col(c), r"\|", ""))
        silver_salesperson_region = silver_salesperson_region.withColumn(c, F.regexp_replace(F.col(c), r"\s-\s", " "))

# 4) Remove rows with null keys
silver_salesperson_region = silver_salesperson_region.filter(
    F.col("employeekey").isNotNull() & F.col("salesterritorykey").isNotNull()
)

# Remove duplicates 
silver_salesperson_region = silver_salesperson_region.dropDuplicates(["employeekey", "salesterritorykey"])

# Save as Silver Delta table
silver_salesperson_region.write
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .format("delta")
    .saveAsTable("workspace.default.silver_salesperson_region")

# Preview 
display(silver_salesperson_region)

# Silver Table (Targets)

In [0]:
%sql
SELECT * FROM workspace.default.bronze_targets;

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
bronze_targets = spark.table("workspace.default.bronze_targets")

# Trim columns
df = (
    bronze_targets
    .withColumn("target", F.trim("target"))
    .withColumn("targetmonth", F.trim("targetmonth"))
)

# Extract Currency

df = df.withColumn(
    "currency",
    F.when(F.col("target").rlike("\\$"), F.lit("USD"))
     .when(F.col("target").rlike("€"), F.lit("EUR"))
     .when(F.col("target").rlike("£"), F.lit("GBP"))
)

# Clean numeric value
# remove symbols
clean_num = F.regexp_replace(F.col("target"), "[\\$€£\\s]", "")

# remove thousand separators
clean_num = F.regexp_replace(clean_num, "\\.", "")
clean_num = F.regexp_replace(clean_num, ",", "")

df = df.withColumn(
    "target_value",
    clean_num.cast("double")
)

# remove original column
df = df.drop("target")

# Parse Month
df = df.withColumn(
    "targetmonth_clean",
    F.regexp_replace("targetmonth", "^[A-Za-z]+,\\s*", "")
)

df = df.withColumn(
    "target_month",
    F.to_date("targetmonth_clean", "MMMM d, yyyy")
)

df = df.drop("targetmonth", "targetmonth_clean")


# Remove null keys
df = df.filter(F.col("employeeid").isNotNull())

# Remove duplicates

df = df.dropDuplicates(["employeeid", "target_month"])


# Reorder columns

df = df.select(
    "employeeid",
    "target_value",
    "currency",
    "target_month",
    "ingestion_timestamp"
)

# Save Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.default.silver_targets")

# Preview
display(df.orderBy("employeeid", "target_month"))

# Silver Table (Salesperson)

In [0]:
%sql
SELECT * FROM workspace.default.bronze_salesperson;

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
bronze_salesperson = spark.table("workspace.default.bronze_salesperson")

# Trim string columns (safe)
df = (bronze_salesperson.withColumn("employeekey", F.trim(F.col("employeekey"))).withColumn("salesperson", F.trim(F.col("salesperson"))).withColumn("title", F.trim(F.col("title"))).withColumn("upn", F.trim(F.col("upn"))))

df = df.withColumn( "employeeid",F.expr("try_cast(nullif(trim(cast(employeeid as string)), '') as bigint)"))

# Normalize UPN (email) to lowercase plus null if empty after trim
df = df.withColumn("upn",F.when(F.col("upn").isNull() | (F.col("upn") == ""), None).otherwise(F.lower(F.col("upn"))))

# Remove rows with null key
df = df.filter(F.col("employeeid").isNotNull())

# Remove duplicates (natural key = employeeid)
df = df.dropDuplicates(["employeeid"])

# Reorder columns 
df = df.select(
    "employeekey",
    "employeeid",
    "salesperson",
    "title",
    "upn",
    "ingestion_timestamp"
)

# Save Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.default.silver_salesperson")

# Preview 
display(df)

# Silver Table (Reseller)

In [0]:
%sql
SELECT * FROM workspace.default.bronze_reseller;

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.silver_reseller;

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
df = spark.table("workspace.default.bronze_reseller")

# Universal string cleaning (for ALL string columns)
# trim
# replace known "missing" tokens with NULL
string_cols = [c for c, t in df.dtypes if t == "string"]

for c in string_cols:
    cleaned = F.trim(F.col(c))
    df = df.withColumn(
        c,
        F.when(
            cleaned.isNull() |
            (cleaned == "") |
            F.upper(cleaned).isin("N/A", "NA", "NULL", "NONE", "-"),
            None
        ).otherwise(cleaned)
    )

# Fix country values 
if "country_region" in df.columns:
    # Normalize country to a letters-only uppercase token:
    # trim
    # replace NBSP with normal space
    # remove ALL non-letters (dots, spaces, newlines, etc.)
    norm = F.upper(
        F.regexp_replace(
            F.regexp_replace(F.trim(F.col("country_region")), u"\u00A0", " "),
            r"[^A-Za-z]", ""
        )
    )

    df = df.withColumn(
        "country_region",
        F.when(F.col("country_region").isNull(), None)

         # UK variants (UK, U.K, U.K., U K, UK\n, etc.) to United Kingdom
         .when(norm == F.lit("UK"), F.lit("United Kingdom"))

         # "United Kingdom" written in any case / punctuation to United Kingdom
         .when(norm == F.lit("UNITEDKINGDOM"), F.lit("United Kingdom"))

         # Common typo variants to United Kingdom
         .when(F.lower(F.trim(F.col("country_region"))).isin("united kindom", "unitedkindom"), F.lit("United Kingdom"))

         .otherwise(F.trim(F.col("country_region")))
    )

# employeeid safe cast (handles '', '   ', malformed) if column exists
if "employeeid" in df.columns:
    df = df.withColumn(
        "employeeid",
        F.expr("try_cast(nullif(trim(cast(employeeid as string)), '') as bigint)")
    )

# Normalize email/upn if present (lowercase)
if "upn" in df.columns:
    df = df.withColumn("upn", F.lower(F.col("upn")))

if "email" in df.columns:
    df = df.withColumn("email", F.lower(F.col("email")))

# Remove rows with null key (prefer resellerid/resellerkey, else employeeid)
key_cols = []
if "resellerid" in df.columns:
    key_cols.append("resellerid")
if "resellerkey" in df.columns:
    key_cols.append("resellerkey")
if not key_cols and "employeeid" in df.columns:
    key_cols.append("employeeid")

if len(key_cols) == 1:
    df = df.filter(F.col(key_cols[0]).isNotNull())
elif len(key_cols) > 1:
    df = df.filter(F.coalesce(*[F.col(k) for k in key_cols]).isNotNull())

# Remove duplicates (best natural key available)
if "resellerid" in df.columns:
    df = df.dropDuplicates(["resellerid"])
elif "resellerkey" in df.columns:
    df = df.dropDuplicates(["resellerkey"])
elif "employeeid" in df.columns:
    df = df.dropDuplicates(["employeeid"])
else:
    df = df.dropDuplicates()

# Ensure ingestion_timestamp exists
if "ingestion_timestamp" not in df.columns:
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())

# Reorder columns: ingestion_timestamp ALWAYS last
cols_no_ing = [c for c in df.columns if c != "ingestion_timestamp"]
df = df.select(*cols_no_ing, "ingestion_timestamp")

# Save Silver table
(
    df.write
      .mode("overwrite")
      .option("overwriteSchema", "true")
      .format("delta")
      .saveAsTable("workspace.default.silver_reseller")
)

# Preview
display(df)

In [0]:
%sql
SELECT *
FROM workspace.default.silver_reseller
WHERE country_region = 'UK'
LIMIT 50;

# Silver Table (Product)

In [0]:
%sql
SELECT * FROM workspace.default.bronze_product;

In [0]:
from pyspark.sql import functions as F

# Load Bronze table
df = spark.table("workspace.default.bronze_product")

# Universal string cleaning:
#    - remove '|'
#    - trim
#    - map "N/A", "NA", "NULL", "NONE", "-", "" -> NULL
string_cols = [c for c, t in df.dtypes if t == "string"]

for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df = df.withColumn(
        c,
        F.when(
            F.col(c).isNull(), None
        ).when(
            F.upper(cleaned).isin("N/A", "NA", "NULL", "NONE", "-", ""), None
        ).otherwise(cleaned)
    )

# Standardize color values (if column exists)
if "color" in df.columns:
    df = df.withColumn(
        "color",
        F.when(F.col("color").isNull(), None)
         .when(F.lower(F.col("color")) == F.lit("multi"), F.lit("Multi"))
         .otherwise(F.initcap(F.lower(F.col("color"))))
    )

# Split standard_cost to currency and standard_cost_value
if "standard_cost" in df.columns:

    df = df.withColumn(
        "currency",
        F.when(F.col("standard_cost").rlike("\\$"), F.lit("USD"))
         .when(F.col("standard_cost").rlike("€"), F.lit("EUR"))
         .when(F.col("standard_cost").rlike("£"), F.lit("GBP"))
         .otherwise(None)
    )

    num = F.regexp_replace(F.col("standard_cost"), r"[\$€£\s]", "")
    num = F.regexp_replace(num, ",", "")

    num = F.when(
        num.rlike(r"^\d{1,3}(\.\d{3})+$"),
        F.regexp_replace(num, r"\.", "")
    ).otherwise(num)

    df = df.withColumn("sc_tmp", num)
    df = df.withColumn(
        "standard_cost_value",
        F.expr("try_cast(nullif(trim(cast(sc_tmp as string)), '') as double)")
    ).drop("sc_tmp")

    df = df.drop("standard_cost")

# Remove null keys and duplicates 
df = df.filter(F.col("productkey").isNotNull())
df = df.dropDuplicates(["productkey"])

# Ensure ingestion_timestamp exists
if "ingestion_timestamp" not in df.columns:
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())

# Reorder columns 
preferred_order = []
for c in ["productkey", "product", "standard_cost_value", "currency", "color"]:
    if c in df.columns:
        preferred_order.append(c)

rest = [c for c in df.columns if c not in preferred_order and c != "ingestion_timestamp"]
df = df.select(*preferred_order, *rest, "ingestion_timestamp")

# Save Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.default.silver_product")

# Preview
display(df.orderBy("productkey"))

In [0]:
%sql
SHOW TABLES IN workspace.default;

# Silver Table (Sales)(Merged)

In [0]:
%sql
SELECT * FROM workspace.default.bronze_sales_2017;

In [0]:
from pyspark.sql import functions as F

# Load bronze tables
s17 = spark.table("workspace.default.bronze_sales_2017")
s18 = spark.table("workspace.default.bronze_sales_2018")
s19 = spark.table("workspace.default.bronze_sales_2019")
s20 = spark.table("workspace.default.bronze_sales_2020")

# Merge (safe with schema differences)
df = (
    s17.unionByName(s18, allowMissingColumns=True)
       .unionByName(s19, allowMissingColumns=True)
       .unionByName(s20, allowMissingColumns=True)
)

# Universal string cleaning
string_cols = [c for c, t in df.dtypes if t == "string"]
for c in string_cols:
    cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
    df = df.withColumn(
        c,
        F.when(
            cleaned.isNull()
            | (cleaned == "")
            | F.upper(cleaned).isin("N/A", "NA", "NULL", "NONE", "-"),
            None
        ).otherwise(cleaned)
    )

# Parse orderdate if it is weekday format
if "orderdate" in df.columns:
    cleaned_date = F.regexp_replace(F.col("orderdate").cast("string"), r"^[A-Za-z]+,\s*", "")
    df = df.withColumn("orderdate", F.to_date(cleaned_date, "MMMM d, yyyy"))

# Create currency from sales symbol 
if "sales" in df.columns and "currency" not in df.columns:
    df = df.withColumn(
        "currency",
        F.when(F.col("sales").cast("string").rlike("\\$"), F.lit("USD"))
         .when(F.col("sales").cast("string").rlike("€"), F.lit("EUR"))
         .when(F.col("sales").cast("string").rlike("£"), F.lit("GBP"))
         .otherwise(None)
    )

# Convert money columns to numeric (remove symbols and commas)
def to_double_money(colname: str):
    return (
        F.regexp_replace(F.col(colname).cast("string"), r"[\$€£]", "")
         .transform(lambda x: F.regexp_replace(x, r",", ""))
         .cast("double")
    )

# unit_price
if "unit_price" in df.columns:
    df = df.withColumn("unit_price", F.regexp_replace(F.col("unit_price").cast("string"), r"[\$€£,]", "").cast("double"))

# cost
if "cost" in df.columns:
    df = df.withColumn("cost", F.regexp_replace(F.col("cost").cast("string"), r"[\$€£,]", "").cast("double"))

# sales
if "sales" in df.columns:
    df = df.withColumn("sales", F.regexp_replace(F.col("sales").cast("string"), r"[\$€£,]", "").cast("double"))

# Drop sales_value 
if "sales_value" in df.columns:
    df = df.drop("sales_value")

# Drop duplicates (natural key)
if "salesordernumber" in df.columns:
    df = df.dropDuplicates(["salesordernumber"])
else:
    df = df.dropDuplicates()

# Ensure ingestion_timestamp exists
if "ingestion_timestamp" not in df.columns:
    df = df.withColumn("ingestion_timestamp", F.current_timestamp())

# Force ingestion_timestamp last
cols_no_ing = [c for c in df.columns if c != "ingestion_timestamp"]
df = df.select(*cols_no_ing, "ingestion_timestamp")

# Save Silver table
df.write.mode("overwrite").option("overwriteSchema", "true").format("delta").saveAsTable("workspace.default.silver_sales")

# Preview
display(df)

In [0]:
%sql
DROP TABLE IF EXISTS workspace.default.silver_targets;
DROP TABLE IF EXISTS workspace.default.silver_salesperson;
DROP TABLE IF EXISTS workspace.default.silver_reseller;
DROP TABLE IF EXISTS workspace.default.silver_product;
DROP TABLE IF EXISTS workspace.default.silver_sales;

# Final Script

In [0]:
from pyspark.sql import functions as F

# ============================================================
# SILVER MASTER PIPELINE (Bronze to Silver)
# Tables covered:
# - bronze_targets             - silver_targets
# - bronze_salesperson         - silver_salesperson
# - bronze_reseller            - silver_reseller
# - bronze_product             - silver_product
# - bronze_sales_2017...2020   - silver_sales
#
# Global Silver Rules:
# Trim strings
# Remove pipes "|"
# Replace N/A, NA, NULL, NONE, "-", "" - NULL
# Standardize country_region: UK / U.K / U.K. / U K / typos - United Kingdom 
# USA / U.S.A / U.S. / U S A / typos - United States
# Monetary columns - numeric values and currency column when needed
# Remove duplicates and null keys
# ingestion_timestamp ALWAYS last
# ============================================================

# Helpers

MISSING_TOKENS = {"N/A", "NA", "NULL", "NONE", "-", ""}

def clean_all_strings(df):
    """Trim, remove pipes, and map missing tokens to NULL across all string columns."""
    string_cols = [c for c, t in df.dtypes if t == "string"]
    for c in string_cols:
        cleaned = F.trim(F.regexp_replace(F.col(c), r"\|", ""))
        df = df.withColumn(
            c,
            F.when(
                cleaned.isNull() |
                (cleaned == "") |
                F.upper(cleaned).isin(*list(MISSING_TOKENS)),
                None
            ).otherwise(cleaned)
        )
    return df


def ensure_ingestion_last(df):
    """Ensure ingestion_timestamp exists and is the last column."""
    if "ingestion_timestamp" not in df.columns:
        df = df.withColumn("ingestion_timestamp", F.current_timestamp())
    cols_no_ing = [c for c in df.columns if c != "ingestion_timestamp"]
    return df.select(*cols_no_ing, "ingestion_timestamp")


def detect_currency_from_symbol(col):
    """Return a Column with currency code based on symbol in the given column."""
    return (
        F.when(col.cast("string").rlike("\\$"), F.lit("USD"))
         .when(col.cast("string").rlike("€"), F.lit("EUR"))
         .when(col.cast("string").rlike("£"), F.lit("GBP"))
         .otherwise(None)
    )


def money_to_double(df, colname):
    """
    Convert a money string column to double by removing currency symbols and thousand separators.
    Keeps the column name the same.
    """
    if colname in df.columns:
        df = df.withColumn(
            colname,
            F.regexp_replace(F.col(colname).cast("string"), r"[\$€£,]", "").cast("double")
        )
    return df


def split_money_to_value_and_currency(df, source_col, value_col, currency_col="currency", drop_source=True):
    """
    Split a money string column into:
      - value_col (double)
      - currency_col (USD/EUR/GBP) if not already present
    """
    if source_col not in df.columns:
        return df

    if currency_col not in df.columns:
        df = df.withColumn(currency_col, detect_currency_from_symbol(F.col(source_col)))

    df = df.withColumn(value_col, F.regexp_replace(F.col(source_col).cast("string"), r"[\$€£,]", "").cast("double"))

    if drop_source:
        df = df.drop(source_col)

    return df


def parse_weekday_date(df, source_col, out_col):
    """
    Parse "Thursday, October 1, 2020" into DATE.
    Removes weekday prefix then uses to_date with "MMMM d, yyyy".
    """
    if source_col not in df.columns:
        return df

    cleaned = F.regexp_replace(F.col(source_col).cast("string"), r"^[A-Za-z]+,\s*", "")
    df = df.withColumn(out_col, F.to_date(cleaned, "MMMM d, yyyy"))
    return df


def fix_country_region_uk(df, colname="country_region"):
    """
    Robust mapping:
      UK / U.K / U.K. / U K / hidden chars to United Kingdom
      United Kindom / unitedkindom to United Kingdom
      United Kingdom (any case/punct) to United Kingdom
    """
    if colname not in df.columns:
        return df

    col_trim = F.trim(F.col(colname))
    norm = F.upper(
        F.regexp_replace(
            F.regexp_replace(col_trim, u"\u00A0", " "),
            r"[^A-Za-z]",
            ""
        )
    )

    df = df.withColumn(
        colname,
        F.when(col_trim.isNull(), None)
         .when(norm == F.lit("UK"), F.lit("United Kingdom"))
         .when(norm == F.lit("UNITEDKINGDOM"), F.lit("United Kingdom"))
         .when(F.lower(col_trim).isin("united kindom", "unitedkindom"), F.lit("United Kingdom"))
         .otherwise(col_trim)
    )
    return df


def standardize_color(df, colname="color"):
    """Normalize color formatting: 'YELLOW'/'yellow' to 'Yellow', 'multi' to 'Multi'."""
    
    if colname not in df.columns:
        return df

    df = df.withColumn(
        colname,
        F.when(F.col(colname).isNull(), None)
         .when(F.lower(F.col(colname)) == F.lit("multi"), F.lit("Multi"))
         .otherwise(F.initcap(F.lower(F.col(colname))))
    )
    return df


# Builders (one per Silver table)

def build_silver_targets():
    df = spark.table("workspace.default.bronze_targets")

    # apply standard cleaning
    df = clean_all_strings(df)

    # detect currency from symbol in target
    if "target" in df.columns:
        df = df.withColumn("currency", detect_currency_from_symbol(F.col("target")))

        # convert target to numeric
        df = df.withColumn(
            "target_value",
            F.regexp_replace(F.col("target").cast("string"), r"[\$€£,]", "").cast("double")
        )

        df = df.drop("target")

    # parse targetmonth -> target_month
    if "targetmonth" in df.columns:
        df = parse_weekday_date(df, "targetmonth", "target_month")
        df = df.drop("targetmonth")

    # remove null keys + duplicates
    if "employeeid" in df.columns:
        df = df.filter(F.col("employeeid").isNotNull())

        if "target_month" in df.columns:
            df = df.dropDuplicates(["employeeid", "target_month"])
        else:
            df = df.dropDuplicates(["employeeid"])

    # preferred column order
    preferred = [c for c in ["employeeid", "target_value", "currency", "target_month"] if c in df.columns]
    rest = [c for c in df.columns if c not in preferred and c != "ingestion_timestamp"]
    df = df.select(*preferred, *rest)

    # ingestion_timestamp last
    df = ensure_ingestion_last(df)

    (
        df.write
          .mode("overwrite")
          .option("overwriteSchema", "true")
          .format("delta")
          .saveAsTable("workspace.default.silver_targets")
    )
    

    
def build_silver_salesperson():
    df = spark.table("workspace.default.bronze_salesperson")
    df = clean_all_strings(df)

    # employeeid safe cast (serverless safe)
    if "employeeid" in df.columns:
        df = df.withColumn("employeeid", F.expr("try_cast(nullif(trim(cast(employeeid as string)), '') as bigint)"))

    # normalize upn/email lowercase
    if "upn" in df.columns:
        df = df.withColumn("upn", F.lower(F.col("upn")))
    if "email" in df.columns:
        df = df.withColumn("email", F.lower(F.col("email")))

    # remove null key + dedup
    if "employeeid" in df.columns:
        df = df.filter(F.col("employeeid").isNotNull()).dropDuplicates(["employeeid"])

    # reorder: ingestion last
    df = ensure_ingestion_last(df)

    (df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.default.silver_salesperson"))


def build_silver_reseller():
    df = spark.table("workspace.default.bronze_reseller")
    df = clean_all_strings(df)

    # fix country column name is country_region in your data
    df = fix_country_region_uk(df, "country_region")

    # remove null key and dedup: resellerkey is natural key
    if "resellerkey" in df.columns:
        df = df.filter(F.col("resellerkey").isNotNull()).dropDuplicates(["resellerkey"])

    df = ensure_ingestion_last(df)

    (df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.default.silver_reseller"))


def build_silver_product():
    df = spark.table("workspace.default.bronze_product")
    df = clean_all_strings(df)

    # standardize color
    df = standardize_color(df, "color")

    # standard_cost - standard_cost_value to currency, drop standard_cost
    df = split_money_to_value_and_currency(
        df,
        source_col="standard_cost",
        value_col="standard_cost_value",
        currency_col="currency",
        drop_source=True
    )

    # remove null key to dedup: productkey
    if "productkey" in df.columns:
        df = df.filter(F.col("productkey").isNotNull()).dropDuplicates(["productkey"])

    # reorder preferred to ingestion last
    preferred = [c for c in ["productkey", "product", "standard_cost_value", "currency", "color"] if c in df.columns]
    rest = [c for c in df.columns if c not in preferred and c != "ingestion_timestamp"]
    df = df.select(*preferred, *rest)
    df = ensure_ingestion_last(df)

    (df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.default.silver_product"))


def build_silver_sales():
    s17 = spark.table("workspace.default.bronze_sales_2017")
    s18 = spark.table("workspace.default.bronze_sales_2018")
    s19 = spark.table("workspace.default.bronze_sales_2019")
    s20 = spark.table("workspace.default.bronze_sales_2020")

    df = (
        s17.unionByName(s18, allowMissingColumns=True)
           .unionByName(s19, allowMissingColumns=True)
           .unionByName(s20, allowMissingColumns=True)
    )

    df = clean_all_strings(df)

    # parse orderdate (if present)
    df = parse_weekday_date(df, "orderdate", "orderdate")

    # currency from sales (if sales exists and currency missing)
    if "sales" in df.columns and "currency" not in df.columns:
        df = df.withColumn("currency", detect_currency_from_symbol(F.col("sales")))

    # convert money columns to numeric
    df = money_to_double(df, "unit_price")
    df = money_to_double(df, "cost")
    df = money_to_double(df, "sales")

    # remove redundant sales_value if exists
    if "sales_value" in df.columns:
        df = df.drop("sales_value")

    # dedup by salesordernumber if present
    if "salesordernumber" in df.columns:
        df = df.dropDuplicates(["salesordernumber"])
    else:
        df = df.dropDuplicates()

    df = ensure_ingestion_last(df)

    (df.write.mode("overwrite")
        .option("overwriteSchema", "true")
        .format("delta")
        .saveAsTable("workspace.default.silver_sales"))



# RUN ALL SILVER BUILDERS
build_silver_targets()
build_silver_salesperson()
build_silver_reseller()
build_silver_product()
build_silver_sales()

print("Silver pipeline completed: targets, salesperson, reseller, product, sales")

In [0]:
%sql
SHOW TABLES IN workspace.default;

In [0]:
%sql
SELECT * FROM  workspace.default.silver_region